In [10]:
from pathlib import Path
import json
import numpy as np
import cv2
import shutil
from PIL import Image
from tqdm import tqdm

# =========================
# CONFIG: POLISH ONLY
# =========================

IMAGES_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/dataset/nail_polish"
)

JSON_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/excluded_categorized/nail_polish"
)

OUT_DIR = Path(
    "/Users/williamtsai/Desktop/nail_unusable_classifier/dataset/nail_polish_nail_only_png"
)

TARGET_LABELS = {"nail"}       # only use nail label
SAVE_MASKS = False
KEEP_ORIGINAL_BG = False
MIN_PIXELS = 50

# reset output
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("IMAGES_DIR:", IMAGES_DIR, IMAGES_DIR.exists())
print("JSON_DIR  :", JSON_DIR, JSON_DIR.exists())
print("OUT_DIR   :", OUT_DIR)

if not IMAGES_DIR.exists():
    raise FileNotFoundError(IMAGES_DIR)

if not JSON_DIR.exists():
    raise FileNotFoundError(JSON_DIR)

IMAGES_DIR: /Users/williamtsai/Desktop/nail_unusable_classifier/dataset/nail_polish True
JSON_DIR  : /Users/williamtsai/Desktop/nail_unusable_classifier/excluded_categorized/nail_polish True
OUT_DIR   : /Users/williamtsai/Desktop/nail_unusable_classifier/dataset/nail_polish_nail_only_png


In [11]:
def find_image_for_json(json_path: Path, images_dir: Path):
    """
    Match JSON filename stem to an image with .jpg/.jpeg/.png.
    Same logic as your old working notebook, but recursive.
    """
    stem = json_path.stem

    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        # first: same direct folder style
        cand = images_dir / f"{stem}{ext}"
        if cand.exists():
            return cand

    # second: recursive search, useful if images are nested
    for ext in (".png", ".jpg", ".jpeg", ".PNG", ".JPG", ".JPEG"):
        matches = list(images_dir.rglob(f"{stem}{ext}"))
        if matches:
            return matches[0]

    return None


def polygon_to_mask(shape_points, h, w):
    pts = np.array(shape_points, dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def tight_bbox_from_mask(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    return int(x0), int(y0), int(x1), int(y1)


def apply_mask_and_crop(img_bgr, mask01):
    bbox = tight_bbox_from_mask(mask01)

    if bbox is None:
        return None, None, None

    x0, y0, x1, y1 = bbox
    m255 = (mask01.astype(np.uint8) * 255)

    if KEEP_ORIGINAL_BG:
        masked = img_bgr.copy()
        masked[mask01 == 0] = 0
    else:
        masked = cv2.bitwise_and(img_bgr, img_bgr, mask=m255)

    crop = masked[y0:y1 + 1, x0:x1 + 1]
    crop_mask = m255[y0:y1 + 1, x0:x1 + 1]

    return crop, crop_mask, bbox

In [12]:
json_files = sorted(JSON_DIR.glob("*.json"))
print("Found JSON:", len(json_files))

exported = 0
missing_images = 0
skipped_no_label = 0
skipped_tiny = 0
failed = []

for jp in tqdm(json_files):
    img_path = find_image_for_json(jp, IMAGES_DIR)

    if img_path is None:
        missing_images += 1
        continue

    try:
        data = json.loads(jp.read_text(encoding="utf-8"))

        img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)

        if img_bgr is None:
            missing_images += 1
            continue

        h, w = img_bgr.shape[:2]
        shapes = data.get("shapes", [])

        # only use nail label
        nail_shapes = [
            s for s in shapes
            if str(s.get("label", "")).lower().strip() == "nail"
            and len(s.get("points", [])) >= 3
        ]

        if not nail_shapes:
            skipped_no_label += 1
            continue

        for k, s in enumerate(nail_shapes, start=1):
            nail_mask = polygon_to_mask(s["points"], h, w)

            if int(nail_mask.sum()) < MIN_PIXELS:
                skipped_tiny += 1
                continue

            crop_bgr, crop_mask, bbox = apply_mask_and_crop(img_bgr, nail_mask)

            if crop_bgr is None:
                skipped_tiny += 1
                continue

            # FLAT OUTPUT: no subfolders
            if len(nail_shapes) == 1:
                out_name = f"{jp.stem}.png"
                mask_name = f"{jp.stem}_mask.png"
            else:
                out_name = f"{jp.stem}_nail_{k:02d}.png"
                mask_name = f"{jp.stem}_nail_{k:02d}_mask.png"

            out_img = OUT_DIR / out_name
            cv2.imwrite(str(out_img), crop_bgr)

            if SAVE_MASKS:
                out_m = OUT_DIR / mask_name
                cv2.imwrite(str(out_m), crop_mask)

            exported += 1

    except Exception as e:
        failed.append((jp.name, str(e)))

print("\nDone.")
print("JSON files       :", len(json_files))
print("exported         :", exported)
print("missing_images   :", missing_images)
print("skipped_no_label :", skipped_no_label)
print("skipped_tiny     :", skipped_tiny)
print("failed           :", len(failed))
print("OUT_DIR          :", OUT_DIR)

if failed:
    print("\nFirst failed:")
    for name, err in failed[:10]:
        print(name, "->", err)

Found JSON: 95


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 95/95 [00:12<00:00,  7.41it/s]


Done.
JSON files       : 95
exported         : 402
missing_images   : 0
skipped_no_label : 1
skipped_tiny     : 0
failed           : 0
OUT_DIR          : /Users/williamtsai/Desktop/nail_unusable_classifier/dataset/nail_polish_nail_only_png
